# **Práctica 7: SparkML**
# **Alumno: Salvador Calderón Martínez**



1. Cargar los datos especificados a continuación.
2. Realizar el join entre todos los archivos.
3. Verificar si existen duplicados, si existen, eliminarlos.
4. Verificar si existen valores vacíos, si existen, eliminarlos.
5. Contar cuantas estaciones registran ecosistemas:
   * Dañados
   * En riesgo
   * Saludable
4. Agrupar por región y contar.
5. ¿Cuál es el máximo porcentaje del fondo marino que está cubierto por coral vivo (indice_cobertura_coral)? ¿Y el mínimo? ¿En dónde se localiza cada uno de ellos?
6. ¿Cuál es la temperatura promedio registrada en las estaciones del Golfo de California?


# **PySpark**

## Instalación

In [1]:
# Instalar PySpark
!pip install -q findspark pyspark

In [2]:
!apt-get update -qq > /dev/null
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
# Descargar Apache Spark
!wget -q https://dlcdn.apache.org/spark/spark-4.2.0/spark-4.2.0-bin-hadoop3.tgz
!tar xf spark-4.2.0-bin-hadoop3.tgz

In [4]:
# Configurar variables de entorno
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-4.2.0-bin-hadoop3"

## Crear sesión de Spark

In [5]:
import findspark
findspark.init()

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EcosistemasMarinosML") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()


In [7]:
spark.version

'4.2.0'

# **Datos**

Los datos se cargarán directamente de la web (GitHub).

## Cargar datos de las estaciones de monitoreo

In [8]:
import urllib.request

# 1. URL Raw de GitHub
url_estaciones = "https://raw.githubusercontent.com/anaepm/rep/refs/heads/main/estaciones_monitoreo.csv"

# 2. Descargar el archivo
urllib.request.urlretrieve(url_estaciones, "estaciones_monitoreo.csv")

# 3. Leerlo con Spark
df_est = spark.read.option("header", "true").option("inferSchema", "true").csv("estaciones_monitoreo.csv")
df_est.show(5)

+-----------+-------------------+------------------+
|estacion_id|             region|profundidad_metros|
+-----------+-------------------+------------------+
|   EST_0001|Golfo de California|              31.9|
|   EST_0002|       Pacífico Sur|               9.6|
|   EST_0003|       Caribe Norte|              38.1|
|   EST_0004|Golfo de California|              25.5|
|   EST_0005|Golfo de California|              16.8|
+-----------+-------------------+------------------+
only showing top 5 rows


## Cargar datos de las condiciones físicas de los océanos

In [9]:
# 1. URL Raw de GitHub
url_fisicas = "https://raw.githubusercontent.com/anaepm/rep/refs/heads/main/condiciones_fisicas.csv"

# 2. Descargar el archivo
urllib.request.urlretrieve(url_fisicas, "condiciones_fisicas.csv")

# 3. Leerlo con Spark
df_fis = spark.read.option("header", "true").option("inferSchema", "true").csv("condiciones_fisicas.csv")
df_fis.show(5)

+-----------+----------------+----+----------------+---------------+
|estacion_id|temperatura_agua|  ph|oxigeno_disuelto|radiacion_solar|
+-----------+----------------+----+----------------+---------------+
|   EST_0001|           29.49|8.22|            4.16|          362.1|
|   EST_0002|           23.05|8.24|            5.71|          767.3|
|   EST_0003|           24.81| 7.9|            4.05|          678.9|
|   EST_0004|           27.04|8.27|            5.89|          858.2|
|   EST_0005|           27.38|8.19|            8.01|          541.0|
+-----------+----------------+----+----------------+---------------+
only showing top 5 rows


## Cargar datos de las características biológicas de los océanos

In [10]:
# 1. URL Raw de GitHub
url_biologicos = "https://raw.githubusercontent.com/anaepm/rep/refs/heads/main/reportes_biologicos.csv"

# 2. Descargar el archivo
urllib.request.urlretrieve(url_biologicos, "reportes_biologicos.csv")

# 3. Leerlo con Spark
df_bio = spark.read.option("header", "true").option("inferSchema", "true").csv("reportes_biologicos.csv")
df_bio.show(5)

+-----------+--------------------------+----------------------+-----------------------+
|estacion_id|presencia_actividad_humana|indice_cobertura_coral|estado_salud_ecosistema|
+-----------+--------------------------+----------------------+-----------------------+
|   EST_0001|                         1|                 59.89|              En riesgo|
|   EST_0002|                         1|                 65.89|              En riesgo|
|   EST_0003|                         0|                 50.54|              En riesgo|
|   EST_0004|                         1|                 81.08|              Saludable|
|   EST_0005|                         1|                 88.44|              Saludable|
+-----------+--------------------------+----------------------+-----------------------+
only showing top 5 rows


# **Procesamiento y Análisis**

## Unir (join) los tres datasets

In [11]:
# Unimos los tres DataFrames usando 'estacion_id' como llave
df = df_est.join(df_fis, on="estacion_id", how="inner") \
           .join(df_bio, on="estacion_id", how="inner")

print("Filas totales tras el join:", df.count())
df.show(5)


Filas totales tras el join: 780000
+-----------+-------------------+------------------+----------------+----+----------------+---------------+--------------------------+----------------------+-----------------------+
|estacion_id|             region|profundidad_metros|temperatura_agua|  ph|oxigeno_disuelto|radiacion_solar|presencia_actividad_humana|indice_cobertura_coral|estado_salud_ecosistema|
+-----------+-------------------+------------------+----------------+----+----------------+---------------+--------------------------+----------------------+-----------------------+
|   EST_0001|Golfo de California|              31.9|           29.49|8.22|            4.16|          362.1|                         1|                 59.89|              En riesgo|
|   EST_0002|       Pacífico Sur|               9.6|           23.05|8.24|            5.71|          767.3|                         1|                 65.89|              En riesgo|
|   EST_0003|       Caribe Norte|              38.1|   

## Verificar y eliminar duplicados

In [12]:
# Contamos duplicados (filas completas repetidas)
total_filas = df.count()
filas_unicas = df.dropDuplicates().count()
duplicados = total_filas - filas_unicas

print(f"Filas totales: {total_filas}")
print(f"Filas únicas: {filas_unicas}")
print(f"Duplicados encontrados: {duplicados}")

# Eliminamos duplicados si existen
df = df.dropDuplicates()


Filas totales: 780000
Filas únicas: 780000
Duplicados encontrados: 0


## Verificar y eliminar valores vacíos (nulos)

In [13]:
from pyspark.sql.functions import col, sum as spark_sum

# Contamos valores nulos por columna
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

# Eliminamos filas con al menos un valor nulo
filas_antes = df.count()
df = df.dropna()
filas_despues = df.count()

print(f"Filas eliminadas por valores nulos: {filas_antes - filas_despues}")
print(f"Filas finales: {filas_despues}")


+-----------+------+------------------+----------------+---+----------------+---------------+--------------------------+----------------------+-----------------------+
|estacion_id|region|profundidad_metros|temperatura_agua| ph|oxigeno_disuelto|radiacion_solar|presencia_actividad_humana|indice_cobertura_coral|estado_salud_ecosistema|
+-----------+------+------------------+----------------+---+----------------+---------------+--------------------------+----------------------+-----------------------+
|          0|     0|                 0|               0|  0|               0|              0|                         0|                     0|                      0|
+-----------+------+------------------+----------------+---+----------------+---------------+--------------------------+----------------------+-----------------------+

Filas eliminadas por valores nulos: 0
Filas finales: 780000


## Contar estaciones por estado de salud del ecosistema

In [14]:
df.groupBy("estado_salud_ecosistema").count().orderBy("count", ascending=False).show()


+-----------------------+------+
|estado_salud_ecosistema| count|
+-----------------------+------+
|              En riesgo|419056|
|              Saludable|360650|
|                 Dañado|   294|
+-----------------------+------+



## Agrupar por región y contar

In [15]:
df.groupBy("region").count().orderBy("count", ascending=False).show()


+-------------------+------+
|             region| count|
+-------------------+------+
|       Caribe Norte|195324|
|       Pacífico Sur|194950|
|Golfo de California|194888|
|         Caribe Sur|194838|
+-------------------+------+



### (Extra) Estado del ecosistema por región

In [16]:
df.groupBy("region", "estado_salud_ecosistema").count() \
  .orderBy("region", "estado_salud_ecosistema").show(20)


+-------------------+-----------------------+------+
|             region|estado_salud_ecosistema| count|
+-------------------+-----------------------+------+
|       Caribe Norte|                 Dañado|    74|
|       Caribe Norte|              En riesgo|105121|
|       Caribe Norte|              Saludable| 90129|
|         Caribe Sur|                 Dañado|    76|
|         Caribe Sur|              En riesgo|104552|
|         Caribe Sur|              Saludable| 90210|
|Golfo de California|                 Dañado|    83|
|Golfo de California|              En riesgo|104563|
|Golfo de California|              Saludable| 90242|
|       Pacífico Sur|                 Dañado|    61|
|       Pacífico Sur|              En riesgo|104820|
|       Pacífico Sur|              Saludable| 90069|
+-------------------+-----------------------+------+



## Máximo y mínimo porcentaje de cobertura de coral vivo

In [17]:
from pyspark.sql.functions import max as spark_max, min as spark_min

max_coral = df.orderBy(col("indice_cobertura_coral").desc()).first()
min_coral = df.orderBy(col("indice_cobertura_coral").asc()).first()

print("Máxima cobertura de coral vivo:")
print(f"  Estación: {max_coral['estacion_id']} | Región: {max_coral['region']} | Índice: {max_coral['indice_cobertura_coral']}%")

print("\nMínima cobertura de coral vivo:")
print(f"  Estación: {min_coral['estacion_id']} | Región: {min_coral['region']} | Índice: {min_coral['indice_cobertura_coral']}%")


Máxima cobertura de coral vivo:
  Estación: EST_0461 | Región: Caribe Sur | Índice: 98.0%

Mínima cobertura de coral vivo:
  Estación: EST_666817 | Región: Golfo de California | Índice: 32.13%


## Temperatura promedio en el Golfo de California

In [18]:
from pyspark.sql.functions import avg

temp_golfo = df.filter(col("region") == "Golfo de California") \
               .agg(avg("temperatura_agua").alias("temperatura_promedio"))

temp_golfo.show()


+--------------------+
|temperatura_promedio|
+--------------------+
|  26.742909363326547|
+--------------------+



# **SparkML: Regresión lineal — Predicción del índice de cobertura de coral**

Usando el mismo `df` (ya unido, sin duplicados y sin nulos) del análisis anterior, construimos un modelo de regresión lineal para predecir `indice_cobertura_coral` a partir de las condiciones físicas y biológicas de cada estación.

**Aplicando lo aprendido en la práctica anterior (S12), evitamos aquí los mismos errores:**
- Importamos explícitamente `sum` de `pyspark.sql.functions` (como `spark_sum`) para no chocar con el `sum` nativo de Python.
- No reutilizamos `col` como variable de loop.
- Volvemos a revisar nulos después de crear columnas nuevas (no solo al principio).
- Para la variable categórica `region` usamos **One-Hot Encoding** (`StringIndexer` + `OneHotEncoder`) en vez de asignarle números ordinales arbitrarios (1, 2, 3...), ya que las regiones no tienen un orden natural entre sí.
- Excluimos `estado_salud_ecosistema` de las variables predictoras: es una etiqueta derivada directamente del propio `indice_cobertura_coral` (fuga de datos / *data leakage*), así que incluirla haría que el modelo "hiciera trampa".
- Verificamos explícitamente el orden de los atributos en el vector antes de interpretar los coeficientes, en vez de asumirlo a ciegas.


## Preparar los datos para el modelo

In [19]:
from pyspark.sql.functions import col, sum as spark_sum

# 'estacion_id' es solo un identificador (no aporta información predictiva)
# 'estado_salud_ecosistema' se excluye por fuga de datos: es una categoría derivada
# directamente del índice de cobertura de coral que queremos predecir.
df_modelo = df.select(
    "region",
    "profundidad_metros",
    "temperatura_agua",
    "ph",
    "oxigeno_disuelto",
    "radiacion_solar",
    "presencia_actividad_humana",
    "indice_cobertura_coral"
)

# Revisamos nulos en las columnas que vamos a usar (buena práctica: volver a
# verificar después de seleccionar/transformar, no solo al inicio del pipeline)
df_modelo.select(
    [spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_modelo.columns]
).show()


+------+------------------+----------------+---+----------------+---------------+--------------------------+----------------------+
|region|profundidad_metros|temperatura_agua| ph|oxigeno_disuelto|radiacion_solar|presencia_actividad_humana|indice_cobertura_coral|
+------+------------------+----------------+---+----------------+---------------+--------------------------+----------------------+
|     0|                 0|               0|  0|               0|              0|                         0|                     0|
+------+------------------+----------------+---+----------------+---------------+--------------------------+----------------------+



## Codificar la variable categórica `region` (One-Hot Encoding)

In [20]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# StringIndexer convierte las categorías de texto en índices numéricos (0, 1, 2, 3...)
region_indexer = StringIndexer(inputCol="region", outputCol="region_index")

# OneHotEncoder convierte esos índices en un vector binario disperso,
# evitando imponer un orden artificial entre las regiones
region_encoder = OneHotEncoder(inputCol="region_index", outputCol="region_ohe")

df_modelo = region_indexer.fit(df_modelo).transform(df_modelo)
df_modelo = region_encoder.fit(df_modelo).transform(df_modelo)

df_modelo.select("region", "region_index", "region_ohe").distinct().show()


+-------------------+------------+-------------+
|             region|region_index|   region_ohe|
+-------------------+------------+-------------+
|Golfo de California|         2.0|(3,[2],[1.0])|
|       Pacífico Sur|         1.0|(3,[1],[1.0])|
|       Caribe Norte|         0.0|(3,[0],[1.0])|
|         Caribe Sur|         3.0|    (3,[],[])|
+-------------------+------------+-------------+



## Correlación de cada variable numérica con el objetivo

In [21]:
target = "indice_cobertura_coral"
columnas_numericas = ["profundidad_metros", "temperatura_agua", "ph",
                       "oxigeno_disuelto", "radiacion_solar", "presencia_actividad_humana"]

# Usamos 'c' como variable de loop (nunca 'col', para no tapar la función importada)
for c in columnas_numericas:
    corr_value = df_modelo.stat.corr(target, c)
    print(f"Correlación con {c}: {corr_value:.4f}")


Correlación con profundidad_metros: -0.3186
Correlación con temperatura_agua: -0.0798
Correlación con ph: 0.3635
Correlación con oxigeno_disuelto: 0.6132
Correlación con radiacion_solar: 0.0013
Correlación con presencia_actividad_humana: 0.0021


## Ensamblar el vector de atributos

In [22]:
from pyspark.ml.feature import VectorAssembler

atributos = ["profundidad_metros", "temperatura_agua", "ph",
             "oxigeno_disuelto", "radiacion_solar",
             "presencia_actividad_humana", "region_ohe"]

# Verificamos explícitamente el orden antes de usarlo (para no asumirlo a ciegas
# al interpretar los coeficientes más adelante)
print("Orden de atributos en el vector:", atributos)

df_rl = (VectorAssembler(inputCols=atributos, outputCol="Atributos")
    .transform(df_modelo).select("Atributos", "indice_cobertura_coral"))
df_rl.show(5, truncate=False)


Orden de atributos en el vector: ['profundidad_metros', 'temperatura_agua', 'ph', 'oxigeno_disuelto', 'radiacion_solar', 'presencia_actividad_humana', 'region_ohe']
+--------------------------------------------+----------------------+
|Atributos                                   |indice_cobertura_coral|
+--------------------------------------------+----------------------+
|[31.9,29.49,8.22,4.16,362.1,1.0,0.0,0.0,1.0]|59.89                 |
|[9.6,23.05,8.24,5.71,767.3,1.0,0.0,1.0,0.0] |65.89                 |
|[38.1,24.81,7.9,4.05,678.9,0.0,1.0,0.0,0.0] |50.54                 |
|[22.6,26.34,8.09,6.53,575.3,1.0,0.0,1.0,0.0]|81.68                 |
|[9.3,24.95,8.36,7.58,488.1,1.0,1.0,0.0,0.0] |89.29                 |
+--------------------------------------------+----------------------+
only showing top 5 rows


## Entrenamiento y prueba

In [23]:
train_data, test_data = df_rl.randomSplit([0.7, 0.3], seed=0)

print("Filas de entrenamiento:", train_data.count())
print("Filas de prueba:", test_data.count())


Filas de entrenamiento: 545502
Filas de prueba: 234498


In [24]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="Atributos", labelCol="indice_cobertura_coral")
lr_model = lr.fit(train_data)


## Evaluación del modelo

In [25]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = lr_model.transform(test_data)

evaluator_rmse = RegressionEvaluator(labelCol="indice_cobertura_coral", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)
print(f"Error cuadrático medio (RMSE) = {rmse:.4f}")

evaluator_r2 = RegressionEvaluator(labelCol="indice_cobertura_coral", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)
print(f"Coeficiente de determinación (R2) = {r2:.4f}")


Error cuadrático medio (RMSE) = 5.9201
Coeficiente de determinación (R2) = 0.6150


## Coeficientes del modelo

In [26]:
coef = lr_model.coefficients
inter = lr_model.intercept

print(f"Intersección: {inter:.4f}\n")

# Los primeros 6 coeficientes corresponden 1 a 1 con columnas escalares;
# los últimos corresponden a las categorías de 'region_ohe' (una por cada
# categoría, menos la de referencia que absorbe el encoder)
nombres_escalares = ["profundidad_metros", "temperatura_agua", "ph",
                      "oxigeno_disuelto", "radiacion_solar", "presencia_actividad_humana"]

for nombre, valor in zip(nombres_escalares, coef[:len(nombres_escalares)]):
    print(f"  {nombre}: {valor:.4f}")

print("\n  (coeficientes restantes -> categorías de 'region', codificadas por OneHotEncoder)")
for i, valor in enumerate(coef[len(nombres_escalares):]):
    print(f"  region_ohe[{i}]: {valor:.4f}")


Intersección: -107.3150

  profundidad_metros: -0.3000
  temperatura_agua: -0.2735
  ph: 20.0361
  oxigeno_disuelto: 4.4978
  radiacion_solar: 0.0001
  presencia_actividad_humana: -0.0045

  (coeficientes restantes -> categorías de 'region', codificadas por OneHotEncoder)
  region_ohe[0]: -0.0129
  region_ohe[1]: 0.0090
  region_ohe[2]: -0.0141


**Interpretación (a completar según los resultados obtenidos al ejecutar):**

* Revisa el signo y la magnitud de cada coeficiente: un valor positivo indica que, al aumentar esa variable, el modelo predice mayor cobertura de coral; uno negativo, lo contrario.
* Contrasta estos coeficientes con las correlaciones calculadas antes de entrenar el modelo — deberían ser consistentes en signo para las variables más influyentes.
* El R² indica qué proporción de la variabilidad del índice de cobertura de coral es explicada por el modelo; un valor bajo sugiere que factores no incluidos (o relaciones no lineales) también influyen.
